# odl_json_profiler

**Source:** `06_analysis/odl_json_profiler.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Loop through and process items

This cell handles: *Loop through and process items*


In [ ]:
"""ODL JSON Profiler — performance-safe analysis for IKEA PIA itemsummary payloads.

Purpose
-------
- Profile ODL JSON structure WITHOUT schema inference (safe for large files)
- Measure nesting depth, array sizes, and field cardinality
- Generate suggested column_mapping.csv transform_expression entries
- Demonstrate the correct raw-string landing + explicit-schema conformance pattern

Usage
-----
Run in Databricks cluster only. Do NOT run locally — requires Spark + ADLS access.

ODL Performance Model
---------------------
ODL JSON files are large, multi-line, nested objects with embedded arrays.
Simple flatten is slow because:

1. multiLine=true creates one Spark task per file (no intra-file parallelism).
2. Schema inference scans the entire dataset twice before reading data.
3. Exploding arrays (localRetailItems, measurements, media) multiplies rows.
4. select * from inferred StructType reads all columns, even unused ones.
5. JSON has no predicate pushdown; every filter reads the whole file.

Correct ODL pipeline model:

  Landing (Bronze RAW)
  ├── source_format: binaryFile
  ├── land raw bytes as-is, one row per file
  ├── no parsing, no schema inference
  └── partition by ingest_date

  Conformance (Bronze Scalars)
  ├── from_json(CAST(content AS STRING), explicit_schema_ddl)
  ├── extract only top-level scalar fields you need
  ├── NO array explode here
  └── result: flat 1-row-per-document table

  Silver — root entity
  ├── clean, typed scalar fields
  └── merge on primary key

  Silver — per-array child entity (separate source_registry row per array)
  ├── source reads from conformance table (source_type=TABLE or from landing)
  ├── from_json + explode ONE array per entity
  └── separate Delta table per logical entity
"""

from __future__ import annotations


## Section 2: Define `_get_active_spark()` helper function

This cell handles: *Define `_get_active_spark()` helper function*


In [ ]:
def _get_active_spark():
    existing = globals().get("spark")
    if existing is not None:
        return existing
    try:
        from pyspark.sql import SparkSession
    except ImportError as exc:
        raise RuntimeError("PySpark not available. Run in Databricks.") from exc
    session = SparkSession.getActiveSession()
    if session is None:
        raise RuntimeError("No active Spark session. Attach to a cluster.")
    return session


## Section 3: Define `_get_dbutils()` helper function

This cell handles: *Define `_get_dbutils()` helper function*


In [ ]:
def _get_dbutils():
    value = globals().get("dbutils")
    if value is None:
        raise RuntimeError("dbutils not available. Run in Databricks.")
    return value


spark = _get_active_spark()
dbutils = _get_dbutils()

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 1 — Safe file profile (no JSON parsing)
# MAGIC
# MAGIC Read files as binaryFile first. This gives size, count, and timestamps
# MAGIC with zero JSON parsing cost.

# COMMAND ----------

ODL_SOURCE_PATH = (
    "abfss://rngpub@adlsdnapdevbronze.dfs.core.windows.net"
    "/eng511/raw_data/pia/itemsummarypublicprd/"
)

# -- Read as binary: safe, fast, no schema inference
df_raw = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.json")
    .load(ODL_SOURCE_PATH)
)

from pyspark.sql import functions as F

profile = df_raw.select(
    F.count("*").alias("file_count"),
    F.sum("length").alias("total_bytes"),
    F.avg("length").alias("avg_bytes_per_file"),
    F.max("length").alias("max_bytes_per_file"),
    F.min("length").alias("min_bytes_per_file"),
    F.min("modificationTime").alias("oldest_file"),
    F.max("modificationTime").alias("newest_file"),
).collect()[0]

print("=== ODL File Profile (binaryFile, no parsing) ===")
print(f"  Files            : {profile['file_count']:,}")
print(f"  Total size       : {profile['total_bytes'] / 1024 / 1024:.1f} MB")
print(f"  Avg per file     : {profile['avg_bytes_per_file'] / 1024:.1f} KB")
print(f"  Max file size    : {profile['max_bytes_per_file'] / 1024 / 1024:.2f} MB")
print(f"  Oldest file      : {profile['oldest_file']}")
print(f"  Newest file      : {profile['newest_file']}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 2 — Sample ONE file and inspect raw structure
# MAGIC
# MAGIC Read a single file as a string so we can see nesting depth before parsing.

# COMMAND ----------

import json

# Sample the smallest file to avoid large parse cost
sample_row = (
    df_raw
    .orderBy("length")                      # smallest first = cheapest to parse
    .select(F.col("content").cast("string").alias("raw"), "path", "length")
    .limit(1)
    .collect()[0]
)

print(f"Sampling file: {sample_row['path']}")
print(f"Size         : {sample_row['length'] / 1024:.1f} KB")
print()

raw_json = sample_row["raw"]
try:
    parsed = json.loads(raw_json)
    top_keys = list(parsed.keys()) if isinstance(parsed, dict) else (
        list(parsed[0].keys()) if parsed and isinstance(parsed[0], dict) else []
    )
    print("=== Top-level keys ===")
    for k in top_keys:
        v = parsed[k] if isinstance(parsed, dict) else parsed[0][k]
        v_type = type(v).__name__
        if isinstance(v, list):
            v_type = f"list[{type(v[0]).__name__ if v else 'empty'}] len={len(v)}"
        elif isinstance(v, dict):
            v_type = f"dict keys=[{', '.join(list(v.keys())[:5])}...]"
        print(f"  {k:<40} {v_type}")
except Exception as exc:
    print(f"Could not parse JSON: {exc}")
    print("First 500 chars:", raw_json[:500])

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 3 — Nesting and array depth analysis (sample only)
# MAGIC
# MAGIC Analyse a small sample (100 files max) to understand array cardinalities.
# MAGIC Do NOT run on full dataset before schema is known.

# COMMAND ----------

MAX_SAMPLE = 100

df_sample_str = (
    df_raw
    .orderBy("length")
    .limit(MAX_SAMPLE)
    .select(F.col("content").cast("string").alias("raw"))
)


## Section 4: Define `analyse_record()` function with logic for processing

This cell handles: *Define `analyse_record()` function with logic for processing*


In [ ]:
def analyse_record(raw: str | None) -> dict:
    """Return array sizes and depth from a single raw JSON string."""
    if not raw:
        return {}
    try:
        obj = json.loads(raw)
        if isinstance(obj, list) and obj:
            obj = obj[0]
        if not isinstance(obj, dict):
            return {}

        result: dict[str, int] = {}
        for key, val in obj.items():
            if isinstance(val, list):
                result[f"_array_len_{key}"] = len(val)
        return result
    except Exception:
        return {}


records = df_sample_str.collect()
array_stats: dict[str, list[int]] = {}
for row in records:
    info = analyse_record(row["raw"])
    for k, v in info.items():
        array_stats.setdefault(k, []).append(v)

print("=== Array field analysis (sample) ===")
if array_stats:
    for field, sizes in sorted(array_stats.items()):
        avg_size = sum(sizes) / len(sizes)
        max_size = max(sizes)
        pct_non_empty = sum(1 for s in sizes if s > 0) / len(sizes) * 100
        print(f"  {field:<55} avg={avg_size:6.1f}  max={max_size:5}  non-empty={pct_non_empty:.0f}%")
        if avg_size > 10:
            print(f"    ⚠  HIGH CARDINALITY — do NOT explode inline, use separate silver entity")
else:
    print("  No array fields found in sample.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 4 — Generate explicit DDL schema from sample
# MAGIC
# MAGIC Use `schema_of_json` on a single representative string — NOT on the whole dataset.
# MAGIC This is safe because it runs on the driver with a single value.

# COMMAND ----------

# Get the schema from ONE representative record (driver-side, no cluster scan)
try:
    representative_row = df_sample_str.orderBy(F.length("raw").desc()).limit(1).collect()[0]
    representative_json = representative_row["raw"]

    # schema_of_json on a literal — runs in driver, no distributed scan
    schema_ddl = spark.sql(
        f"SELECT schema_of_json('{representative_json[:5000].replace(chr(39), chr(34))}') AS s"
    ).collect()[0]["s"]
    print("=== Inferred DDL schema (from single record) ===")
    print(schema_ddl[:3000])
    print()
    print("=== How to use in source_options_json ===")
    print('  "odl_schema_ddl": "<paste the schema above here>"')
    print()
    print("=== How to use in transform_expression (column_mapping.csv) ===")
    print()
    print("  -- For scalar top-level fields:")
    print("     get_json_object(raw_payload, '$.fieldName')  AS conformance_col")
    print()
    print("  -- For typed scalar:")
    print("     CAST(get_json_object(raw_payload, '$.quantity') AS DECIMAL(18,3)) AS qty")
    print()
    print("  -- For nested struct scalar:")
    print("     get_json_object(raw_payload, '$.salesStatus.status') AS sales_status")
    print()
    print("  -- For arrays (use SEPARATE silver entity — see Step 5):")
    print("     Do NOT do: from_json(raw, schema).bigArray[*].field  <- slow")
    print("     Instead: separate source_registry row with explode() in transform_expression")
except Exception as exc:
    print(f"Schema inference failed: {exc}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 5 — Performance benchmark: inference vs explicit schema
# MAGIC
# MAGIC Compare read time for three approaches on the sample dataset.

# COMMAND ----------

import time

df_binary_sample = df_raw.limit(50)

# Approach A: multiLine json + full inference (current — slowest)
start = time.time()
try:
    df_a = (
        spark.read.format("json")
        .option("multiLine", "true")
        .option("recursiveFileLookup", "true")
        .load(ODL_SOURCE_PATH)
        .limit(100)
    )
    cnt_a = df_a.count()
    col_count_a = len(df_a.columns)
    t_a = time.time() - start
    print(f"A) multiLine json + inference : {t_a:.1f}s  rows={cnt_a:,}  cols={col_count_a}")
except Exception as exc:
    print(f"A) FAILED: {exc}")

# Approach B: binaryFile + get_json_object on scalar only (recommended for landing)
start = time.time()
try:
    df_b = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .load(ODL_SOURCE_PATH)
        .limit(50)
        .select(
            F.col("path"),
            F.col("modificationTime"),
            F.col("content").cast("string").alias("raw_payload"),
        )
    )
    cnt_b = df_b.count()
    t_b = time.time() - start
    print(f"B) binaryFile raw landing     : {t_b:.1f}s  rows={cnt_b:,}  cols=3 (raw string)")
except Exception as exc:
    print(f"B) FAILED: {exc}")

# Approach C: binaryFile + selective get_json_object extraction (recommended for conformance)
start = time.time()
try:
    df_c = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .load(ODL_SOURCE_PATH)
        .limit(50)
        .select(F.col("content").cast("string").alias("raw_payload"))
        .select(
            F.get_json_object("raw_payload", "$.itemNo").alias("item_no"),
            F.get_json_object("raw_payload", "$.itemType").alias("item_type"),
            F.get_json_object("raw_payload", "$.globalSalesStatus").alias("global_sales_status"),
        )
    )
    cnt_c = df_c.count()
    t_c = time.time() - start
    print(f"C) binaryFile + selective json: {t_c:.1f}s  rows={cnt_c:,}  cols=3 (extracted)")
except Exception as exc:
    print(f"C) FAILED: {exc}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 6 — Suggested metadata model for ODL entities
# MAGIC
# MAGIC The framework supports this pattern via existing transform_expression column.
# MAGIC No code change needed — only metadata changes.

# COMMAND ----------

METADATA_DESIGN = """
=============================================================================
 ODL JSON METADATA MODEL — performance-optimized for itemsummarypublicprd
=============================================================================

PRINCIPLE: One landing row per SOURCE FILE (binaryFile, raw string preserved).
           Conformance extracts scalars only using explicit get_json_object.
           Arrays become SEPARATE silver entities (separate source_registry rows).

─────────────────────────────────────────────────────────────────────────────
LANDING LAYER (source_registry row — ingest raw binary, no parsing)
─────────────────────────────────────────────────────────────────────────────
  source_format      : binaryFile
  source_options_json: {"recursiveFileLookup": "true",
                         "pathGlobFilter": "*.json",
                         "file_ingest_mode": "autoloader"}
  landing_table      : {catalog}.{bronze}.pia_odl_itemsummary_raw
  publish_mode       : append
  primary_key        : (none — one row per file)

  Stored columns: path, modificationTime, length, content (raw bytes)
  Result: fast binary landing, no parsing cost, full fidelity preservation

─────────────────────────────────────────────────────────────────────────────
CONFORMANCE (column_mapping.csv — scalars only, explicit JSON path extraction)
─────────────────────────────────────────────────────────────────────────────
  INPUT: CAST(content AS STRING) → named raw_payload by landing engine

  column_mapping rows (transform_expression → conformance_column):

  get_json_object(raw_payload, '$.itemNo')           → item_no
  get_json_object(raw_payload, '$.itemType')         → item_type
  get_json_object(raw_payload, '$.globalSalesStatus')→ global_sales_status
  get_json_object(raw_payload, '$.validFrom')        → valid_from
  get_json_object(raw_payload, '$.validTo')          → valid_to
  get_json_object(raw_payload, '$.productArea.code') → product_area_code
  raw_payload                                        → raw_payload  ← keep for array extraction

  WHY: get_json_object parses JSON lazily per column, skips all other fields.
       No schema inference. No unused column reads. Columnar pushdown preserved.

─────────────────────────────────────────────────────────────────────────────
SILVER — root entity (scalar attributes only)
─────────────────────────────────────────────────────────────────────────────
  silver_table: {catalog}.{silver}.pia_odl_itemsummary
  publish_mode: merge
  merge_key   : item_no
  optimize    : Z-ORDER BY item_no
  partition   : (none for now; add item_type later if needed)

─────────────────────────────────────────────────────────────────────────────
SILVER — child entity per array (SEPARATE source_registry row each)
─────────────────────────────────────────────────────────────────────────────
  Example: localRetailItems array

  source_type        : TABLE               ← reads from conformance table
  source_path/table  : {catalog}.{bronze}.pia_odl_itemsummary_conformance
  conformance_table  : {catalog}.{bronze}.pia_odl_localretailitem_conformance
  silver_table       : {catalog}.{silver}.pia_odl_localretailitem

  transform_expression for explode in column_mapping:
    item_no                                          → item_no  (parent key)
    explode(from_json(raw_payload,
      'array<struct<countryCode:string,
                    salesStatus:string,
                    price:decimal(18,3)>>'
    ))                                               → local_item  (struct col)
    local_item.countryCode                           → country_code
    local_item.salesStatus                           → local_sales_status

  WHY: each array entity gets its own table, Z-ordered on item_no + country_code.
       Merge and query performance are isolated per entity.
       from_json uses EXPLICIT schema (not inference) so it is fast.

─────────────────────────────────────────────────────────────────────────────
 SUMMARY OF PERFORMANCE WINS
─────────────────────────────────────────────────────────────────────────────
  Problem (current)                  Solution (this model)
  ─────────────────────────────────  ─────────────────────────────────────
  multiLine=true, 1 task/file        binaryFile landing, full parallelism
  schema inference scans all data    get_json_object: no inference needed
  full flatten of all columns        selective extraction of needed fields
  all arrays exploded together       one silver entity per array, isolated
  no predicate pushdown on JSON      Parquet/Delta pushdown on silver layer
  merge on wide table (all cols)     merge on narrow scalar table only

=============================================================================
"""

print(METADATA_DESIGN)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 7 — Generate skeleton source_registry rows for ODL entities

# COMMAND ----------

CATALOG = spark.conf.get("spark.databricks.clusterUsageTags.orgId", "eng511_development_bronze")
# Override with your actual catalog
CATALOG = "eng511_development_bronze"
BRONZE = "bronze_dev"
SILVER = "silver_dev"

odl_entities = [
    # (entity_name, array_path_in_json, array_item_schema_ddl, extract_fields)
    ("localretailitem",
     "$.localRetailItems[*]",
     "struct<countryCode:string,salesStatus:string>",
     ["countryCode", "salesStatus"]),
    ("measurement",
     "$.measurements[*]",
     "struct<type:string,value:decimal(18,3),unit:string>",
     ["type", "value", "unit"]),
    ("categorymain",
     "$.categoryPaths[*]",
     "struct<id:string,name:string,level:int>",
     ["id", "name", "level"]),
]

print("=== Suggested additional source_registry entries for ODL array entities ===\n")
for entity, array_path, schema_ddl, fields in odl_entities:
    print(f"entity       : pia_odl_{entity}")
    print(f"source_type  : FILE  (reads from landing via transform)")
    print(f"landing_table: {CATALOG}.{BRONZE}.pia_odl_{entity}_landing")
    print(f"silver_table : {CATALOG}.{SILVER}.pia_odl_{entity}")
    print(f"array_path   : {array_path}")
    print(f"schema_ddl   : {schema_ddl}")
    print("transform_expressions (column_mapping.csv):")
    print(f"  get_json_object(raw_payload, '$.itemNo')  → item_no  (FK to root)")
    for f in fields:
        col = "".join(f"_{c.lower()}" if c.isupper() else c for c in f).lstrip("_")
        print(f"  local_item.{f:<30}→ {col}")
    print()

print("=== Z-order recommendations ===")
print("  pia_odl_itemsummary    : Z-ORDER BY item_no")
print("  pia_odl_localretailitem: Z-ORDER BY item_no, country_code")
print("  pia_odl_measurement    : Z-ORDER BY item_no, type")
print("  pia_odl_categorymain   : Z-ORDER BY item_no, id")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 8 — Millions of small files: root causes and architecture
# MAGIC
# MAGIC HIP delivers thousands of small JSON files per day → millions total.
# MAGIC This is a fundamentally different problem from complex JSON structure.
# MAGIC Both problems interact, making naive ingestion very slow.

# COMMAND ----------

SMALL_FILES_DESIGN = """
=============================================================================
 MILLIONS OF SMALL FILES — ODL / HIP at scale on Azure ADLS
=============================================================================

ROOT CAUSES OF SLOW INGESTION WITH MILLIONS OF FILES
─────────────────────────────────────────────────────
1. DIRECTORY LISTING (O(n) per run)
   Auto Loader default mode lists the entire ADLS directory on every trigger.
   With millions of files this is 10s of minutes of LIST API calls before
   a single byte of data is read.

2. TASK OVERHEAD > DATA WORK
   Each small file → one Spark task. 10,000 files of 5 KB each =
   10,000 tasks processing 5 KB. The task scheduling overhead (seconds each)
   dwarfs the actual read time (milliseconds each).

3. LANDING TABLE FILE FRAGMENTATION
   Every Auto Loader batch appends many small Parquet files to the Delta table.
   Over time: 1M source files → 1M+ small Parquet files in landing Delta.
   Downstream reads scan thousands of small files with high metadata overhead.

4. SCHEMA INFERENCE ACROSS MILLIONS OF FILES
   If inferSchema=true, Auto Loader samples files from the whole dataset on
   first run. With millions of files this triggers massive parallel reads.

SOLUTION ARCHITECTURE FOR HIGH-VOLUME HIP/ODL DELIVERY
─────────────────────────────────────────────────────────────────────────────

LAYER 1: SWITCH FROM DIRECTORY LISTING TO EVENT GRID NOTIFICATION MODE
─────────────────────────────────────────────────────────────────────────
  source_options_json:
    {
      "cloudFiles.useNotifications": "true",          ← event-driven, O(1) per file
      "cloudFiles.maxFilesPerTrigger": "50000",        ← max files per trigger batch
      "cloudFiles.includeExistingFiles": "false",      ← skip historical on new subscription
      "cloudFiles.backfillInterval": "1 day",          ← safety net: full re-scan daily
      "cloudFiles.fetchParallelism": "8",              ← parallel metadata fetch workers
      "cloudFiles.schemaEvolutionMode": "rescue",      ← handle schema drift safely
      "json_parse_mode": "raw_string",                 ← no JSON parsing at landing
      "post_landing_optimize": true,                   ← compact after each trigger
      "landing_partition_columns": "ingest_date",      ← scope OPTIMIZE per day
      "landing_optimize_zorder": "source_system"       ← Z-order key for conformance joins
    }

  Azure prerequisites (one-time, done by platform/IDNAP admins):
    1. Enable ADLS Gen2 hierarchical namespace (already enabled for eng511)
    2. Create Azure Event Grid subscription on the storage account:
         - Filter: blob created events only (Microsoft.Storage.BlobCreated)
         - Filter path prefix: /eng511/raw_data/pia/itemsummarypublicprd/
         - Endpoint: Azure Event Hub (standard tier, 1 TU minimum)
    3. Grant Databricks managed identity: Azure Event Hub Data Receiver role
    4. Databricks auto-creates the subscription on first Auto Loader run
       when cloudFiles.useNotifications=true

  SQL to check if notification mode is active after first run:
    DESCRIBE HISTORY <catalog>.<bronze>.pia_odl_itemsummary_raw;
    -- look for operationParameters.sourceFileNotificationChannelType = "EventHub"

─────────────────────────────────────────────────────────────────────────────
LAYER 2: LAND AS DELTA ROWS (binaryFile) — CONSOLIDATE FILES BY DESIGN
─────────────────────────────────────────────────────────────────────────────
  source_format: binaryFile (already configured via json_parse_mode=raw_string)

  Result: 1 source file → 1 Delta row (raw_payload string column)
  Landing table is wide (path, modificationTime, raw_payload, ingest metadata)
  but shallow (one row = one source document).

  PARTITIONED BY (ingest_date):
    - Add ingest_date as DATE column derived from ingest_ts in landing metadata
    - Enables OPTIMIZE on single-day partitions (fast, scoped)
    - Enables partition pruning: WHERE ingest_date = '2026-04-13'
    - Enables easy data retention: ALTER TABLE DROP PARTITION (ingest_date < ...)

  In landing_engine.py (already wired):
    write_landing(..., partition_columns=["ingest_date"])
    -- ingest_date must be added by add_landing_metadata or a pre-landing transform

─────────────────────────────────────────────────────────────────────────────
LAYER 3: DAILY OPTIMIZE SCHEDULE — COMPACT DELTA SMALL FILES
─────────────────────────────────────────────────────────────────────────────
  Scenario: 50,000 files/day land → 50,000 small Parquet files in Delta landing.
  After 30 days: 1.5M small files. Queries degrade badly.

  Option A: post_landing_optimize=true (already wired, runs after each trigger)
    -- OPTIMIZE <table> ZORDER BY (source_system)
    -- Scoped to current ingest_date partition
    -- Runs inline after each batch; adds ~30s but prevents accumulation

  Option B: Scheduled Databricks job (separate OPTIMIZE notebook, daily at 03:00)
    -- More flexible: can run OPTIMIZE with VACUUM together
    -- Recommended for very high volume (>100K files/day) where inline is too slow

  SQL for scheduled OPTIMIZE:
    -- Run this daily after landing batch completes
    OPTIMIZE eng511_development_bronze.bronze_dev.pia_odl_itemsummary_raw
      WHERE ingest_date = current_date() - 1
      ZORDER BY (source_system);

    VACUUM eng511_development_bronze.bronze_dev.pia_odl_itemsummary_raw
      RETAIN 168 HOURS;  -- 7 days

─────────────────────────────────────────────────────────────────────────────
LAYER 4: SCHEMA TRACKING — ONE LOCATION PER ENTITY (already configured)
─────────────────────────────────────────────────────────────────────────────
  Auto Loader writes learned schema to cloudFiles.schemaLocation per entity.
  For binaryFile format: schema is fixed (path, modificationTime, length, content).
  → Schema inference is effectively disabled; tracking location is a lightweight
    marker file only.

  For json format (if used): schema inferred ONCE, stored in schemaLocation.
  Subsequent runs reuse stored schema → no re-inference overhead.

─────────────────────────────────────────────────────────────────────────────
EXPECTED PERFORMANCE WITH THIS MODEL
─────────────────────────────────────────────────────────────────────────────
  Volume         Without                With this model
  ─────────────  ─────────────────────  ─────────────────────────────────────
  File listing   O(n): minutes/hours    O(1): Event Grid → sub-second
  Per-trigger    All unprocessed files  max maxFilesPerTrigger (50K cap)
  Task overhead  1 task per source file 1 task per Delta partition (scoped)
  Schema cost    Re-inferred every run  Tracked once, reused always
  Downstream     Scan 1M small Parquet  Scan compacted Parquet by date
  OPTIMIZE       Full table (slow)      Scoped to ingest_date partition

─────────────────────────────────────────────────────────────────────────────
RECOMMENDED source_registry.csv ROW FOR ITEMSUMMARYPUBLICPRD AT SCALE
─────────────────────────────────────────────────────────────────────────────
  source_options_json:
  {
    "file_ingest_mode": "autoloader",
    "recursiveFileLookup": "true",
    "pathGlobFilter": "*.json",
    "cloudFiles.useNotifications": "true",
    "cloudFiles.maxFilesPerTrigger": "50000",
    "cloudFiles.maxBytesPerTrigger": "1g",
    "cloudFiles.includeExistingFiles": "false",
    "cloudFiles.backfillInterval": "1 day",
    "cloudFiles.fetchParallelism": "8",
    "cloudFiles.schemaEvolutionMode": "rescue",
    "json_parse_mode": "raw_string",
    "post_landing_optimize": true,
    "landing_partition_columns": "ingest_date",
    "landing_optimize_zorder": "source_system"
  }

  Note: "cloudFiles.includeExistingFiles": "false" means only NEW files after
  the first run are processed. For initial historical backfill, use a separate
  one-time batch job (ingest_file_batch with explicit date range filter).

=============================================================================
"""

print(SMALL_FILES_DESIGN)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 9 — Measure current file count by date and estimate OPTIMIZE benefit

# COMMAND ----------

import datetime

print("=== Estimating small file accumulation in landing table ===")
print()

# Count files per day in source path to understand notification mode impact
try:
    df_by_date = (
        df_raw
        .withColumn("source_date", F.to_date("modificationTime"))
        .groupBy("source_date")
        .agg(
            F.count("*").alias("file_count"),
            F.sum("length").alias("total_bytes"),
            F.avg("length").alias("avg_bytes"),
        )
        .orderBy("source_date", ascending=False)
        .limit(30)
    )
    df_by_date.show(30, truncate=False)

    totals = df_raw.agg(F.count("*").alias("n"), F.sum("length").alias("bytes")).collect()[0]
    days_span = 30  # assume 30 days for projection
    daily_rate = totals["n"] / days_span
    print(f"Estimated daily file rate: {daily_rate:,.0f} files/day")
    if daily_rate > 10000:
        print("⚠  HIGH VOLUME: notification mode is strongly recommended")
        print(f"   At {daily_rate:,.0f} files/day, directory listing adds "
              f"~{daily_rate / 1000 * 2:.0f}s overhead per trigger run.")
    else:
        print(f"   At {daily_rate:,.0f} files/day, directory listing is manageable "
              "but notification mode is still recommended for >30 day retention.")
except Exception as exc:
    print(f"File-by-date analysis failed: {exc}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Step 10 — Check existing landing table file fragmentation

# COMMAND ----------

LANDING_TABLE = f"{CATALOG}.{BRONZE}.pia_odl_itemsummary_raw"

print(f"=== Delta file fragmentation check for {LANDING_TABLE} ===")
try:
    detail = spark.sql(f"DESCRIBE DETAIL {LANDING_TABLE}").collect()[0]
    num_files = detail["numFiles"]
    size_bytes = detail["sizeInBytes"]
    avg_file_kb = (size_bytes / num_files / 1024) if num_files > 0 else 0
    print(f"  numFiles          : {num_files:,}")
    print(f"  sizeInBytes       : {size_bytes / 1024 / 1024:.1f} MB")
    print(f"  avg file size     : {avg_file_kb:.1f} KB")
    if avg_file_kb < 128:
        print("  ⚠  Small files detected. OPTIMIZE is needed.")
        print(f"  Run: OPTIMIZE {LANDING_TABLE} WHERE ingest_date = current_date() - 1")
    else:
        print("  ✓  File sizes are healthy (>128 KB average).")
except Exception as exc:
    print(f"  Table not found or accessible: {exc}")
    print("  Run initialize_framework first to create landing tables.")
